# Chapter 5: Metrics Monitoring and Alerting System
*System Design Interview Volume 2*

## TL;DR

A large-scale metrics monitoring system collects operational metrics via **pull or push** models, buffers them through a **Kafka pipeline**, and stores them in a **time-series database** (e.g., InfluxDB). An **alerting system** evaluates YAML-defined rules against stored data and dispatches notifications through Kafka to email/SMS/PagerDuty channels. **Downsampling** (7 days raw, 30 days at 1-min, 1 year at 1-hr resolution) and **cold storage** manage the massive data volume. Visualization is best handled by off-the-shelf tools like Grafana.

## Requirements

| Type | Requirement | Detail |
|---|---|---|
| Functional | Metric collection | Collect CPU, memory, disk, request count, etc. |
| Functional | Alerting | Detect anomalies, notify via email/SMS/PagerDuty/webhooks |
| Functional | Visualization | Dashboards with graphs and charts (Grafana) |
| Non-functional | Scale | 100M DAU, 1000 server pools, 100 machines each, ~10M metrics |
| Non-functional | Low latency | Fast queries for dashboards and alert evaluation |
| Non-functional | Reliability | Never miss critical alerts |
| Non-functional | Flexibility | Easily integrate new tech in the pipeline |
| Data retention | Tiered | 7 days raw, 30 days 1-min, 1 year 1-hr resolution |

## Estimation: Metrics Volume and Storage

In [ ]:
# Back-of-envelope for a metrics monitoring system
server_pools = 1_000
machines_per_pool = 100
metrics_per_machine = 100
total_metrics = server_pools * machines_per_pool * metrics_per_machine
print(f"Total unique metric series: {total_metrics:,}")

# Write volume
collection_interval_sec = 10
writes_per_sec = total_metrics / collection_interval_sec
print(f"Write rate: {writes_per_sec:,.0f} data points/sec")

# Storage estimation (each data point ~ 16 bytes: 8B timestamp + 8B value)
bytes_per_point = 16
seconds_per_day = 86_400

# Raw (7 days)
raw_points_per_day = writes_per_sec * seconds_per_day
raw_storage_gb = raw_points_per_day * bytes_per_point * 7 / (1024**3)
print(f"\nRaw storage (7 days): {raw_storage_gb:.0f} GB")

# 1-min resolution (30 days)
downsampled_1min = total_metrics * (seconds_per_day / 60) * 30
storage_1min_gb = downsampled_1min * bytes_per_point / (1024**3)
print(f"1-min resolution (30 days): {storage_1min_gb:.0f} GB")

# 1-hr resolution (1 year)
downsampled_1hr = total_metrics * 24 * 365
storage_1hr_gb = downsampled_1hr * bytes_per_point / (1024**3)
print(f"1-hr resolution (1 year): {storage_1hr_gb:.0f} GB")

total_gb = raw_storage_gb + storage_1min_gb + storage_1hr_gb
print(f"\nTotal storage: {total_gb:,.0f} GB ({total_gb/1024:.1f} TB)")

## High-Level Design

```mermaid
graph TB
    subgraph Sources["Metrics Sources"]
        APP[App Servers]
        DB[Databases]
        MQ[Message Queues]
    end

    subgraph Collection["Metrics Collection"]
        PULL[Pull Collectors]
        PUSH[Push Agents]
        SD["Service Discovery<br/>(etcd/ZooKeeper)"]
    end

    subgraph Pipeline["Transmission Pipeline"]
        KAFKA[Kafka]
    end

    subgraph Storage["Data Storage"]
        TSDB["Time-Series DB<br/>(InfluxDB)"]
        CACHE[Query Cache]
    end

    subgraph Alerting["Alerting System"]
        RULES[Alert Rules Cache]
        AM[Alert Manager]
        STORE[Alert Store]
    end

    subgraph Notify["Notification Channels"]
        EMAIL[Email]
        SMS[SMS]
        PD[PagerDuty]
        WH[Webhooks]
    end

    VIZ[Visualization / Grafana]

    APP --> PULL
    DB --> PULL
    MQ --> PUSH
    SD --> PULL
    PULL --> KAFKA
    PUSH --> KAFKA
    KAFKA --> TSDB
    TSDB --> CACHE
    CACHE --> VIZ
    CACHE --> AM
    RULES --> AM
    AM --> STORE
    AM --> KAFKA
    KAFKA --> EMAIL
    KAFKA --> SMS
    KAFKA --> PD
    KAFKA --> WH
```

## Deep Dive

### Time-Series Data Model

Every metric is a time series with three components:

| Component | Example | Purpose |
|---|---|---|
| Metric name | `cpu.load` | Identifies what is measured |
| Labels (tags) | `host:i631, env:prod` | Identifies the specific source |
| Values | `[(t1, 0.62), (t2, 0.71), ...]` | The actual measurements |

**Line protocol format**: `cpu.load host=webserver01,region=us-west 1613707265 54`

Key property: labels must be **low cardinality** (small set of possible values) for efficient indexing.

### Time-Series Database Selection

Why not a general-purpose database?

| Database Type | Problem for Time-Series |
|---|---|
| Relational (MySQL) | Complex SQL for rolling averages; heavy write load |
| NoSQL (Cassandra) | Requires deep expertise for schema design |
| **Time-Series DB** | **Purpose-built: write-optimized, label indexes, custom query language** |

InfluxDB benchmark: 8 cores + 32GB RAM handles **250,000+ writes/sec**.

InfluxDB query (Flux) for exponential moving average:

```
from(db:"telegraf")
  |> range(start:-1h)
  |> filter(fn: (r) => r._measurement == "foo")
  |> exponentialMovingAverage(size:-10s)
```

### Pull vs Push Collection Models

```mermaid
graph LR
    subgraph PullModel["Pull Model (Prometheus)"]
        SD2["Service Discovery"] --> MC2["Metrics Collector"]
        MC2 -->|"GET /metrics"| WS2[Web Servers]
    end

    subgraph PushModel["Push Model (CloudWatch)"]
        WS3[Web Servers] --> AG[Collection Agent]
        AG -->|"push"| LB[Load Balancer]
        LB --> MC3["Metrics Collectors<br/>(auto-scaling)"]
    end
```

| Aspect | Pull | Push |
|---|---|---|
| Debugging | Easy -- browse /metrics endpoint | Harder -- is it network or app? |
| Health check | No response = server down | Ambiguous -- could be network |
| Short-lived jobs | Might miss them (use push gateway) | Natural fit |
| Firewall/NAT | Needs reachability to all endpoints | Works through firewalls |
| Scalability | Consistent hashing for collector assignment | Auto-scaling collector cluster |

### Metrics Transmission Pipeline

A Kafka queue between collectors and the time-series DB:

- **Decouples** collection from storage
- **Prevents data loss** during DB outages
- **Scales** via partitioning by metric name or labels
- **Prioritizes** critical metrics over less important ones

### Alerting System Flow

1. Alert rules defined in **YAML config files**, loaded into cache
2. **Alert manager** periodically queries the time-series DB
3. If threshold violated, create alert event
4. **Filter, merge, deduplicate** alerts (e.g., merge 3 disk_usage events on same instance)
5. Store alert state in **KV store** (Cassandra): inactive, pending, firing, resolved
6. Push to **Kafka** for reliable delivery
7. Alert consumers dispatch to email / SMS / PagerDuty / webhooks

### Downsampling and Data Retention

In [ ]:
# Demonstrate the storage savings from downsampling
raw_interval_sec = 10
retention_policies = [
    ("Raw (7 days)", raw_interval_sec, 7),
    ("1-min (30 days)", 60, 30),
    ("1-hr (1 year)", 3600, 365),
]

total_points = 0
print(f"{'Policy':<20} {'Points/series':<18} {'Interval':<12} {'Retention'}")
print("-" * 70)
for name, interval, days in retention_policies:
    points_per_series = (86_400 / interval) * days
    total_points += points_per_series
    print(f"{name:<20} {points_per_series:>14,.0f}   {interval:>6}s       {days:>4} days")

# Compare to keeping all raw
all_raw_points = (86_400 / raw_interval_sec) * 365
savings_pct = (1 - total_points / all_raw_points) * 100
print(f"\nTotal points per series with downsampling: {total_points:,.0f}")
print(f"Total points per series if all raw: {all_raw_points:,.0f}")
print(f"Storage savings: {savings_pct:.1f}%")

**Additional optimizations:**
- **Double-delta encoding**: store differences of differences (32-bit timestamps reduced to 4 bits)
- **Cold storage**: move old data to cheaper storage (e.g., S3, glacier)

### Visualization

Best handled by **off-the-shelf tools** like Grafana, which integrates natively with InfluxDB, Prometheus, and other time-series databases. Building a custom visualization system is rarely justified.

## Trade-offs

| Dimension | Option A | Option B |
|---|---|---|
| Pull vs Push | Pull: easy debugging, health checks | Push: works through firewalls, short-lived jobs |
| Kafka in pipeline | Reliable, decoupled, loss-tolerant | Added operational complexity |
| No Kafka (Gorilla-style) | Simpler, in-memory writes | Risk of data loss on partial failure |
| Own query service | Flexibility, caching layer | Time-series DB query interface may suffice |
| Build alerting | Full customization | Complex; off-the-shelf (PagerDuty, Grafana alerts) covers most needs |
| Build visualization | Custom dashboards | Grafana is battle-tested and integrates well |
| Fine-grained downsampling | More data retained | Higher storage costs |

## Key Takeaways

1. **Time-series databases** are essential -- general-purpose DBs cannot handle the write volume and query patterns efficiently
2. **Kafka pipeline** between collectors and storage provides reliability and scalability at scale
3. **Pull and push** both have merits; large organizations typically support both
4. **Downsampling** with tiered retention (raw -> 1 min -> 1 hr) reduces storage by over 95%
5. **Alerting** needs deduplication, state management, and reliable notification delivery
6. **Buy vs build**: use Grafana for visualization and consider off-the-shelf alerting systems

## Related Concepts

- [[time-series-data-model]] -- how metrics are structured and stored
- [[time-series-database]] -- specialized storage for write-heavy metric data
- [[pull-vs-push-collection]] -- two approaches to gathering metrics
- [[metrics-transmission-pipeline]] -- Kafka-based buffering between collection and storage
- [[alerting-system]] -- rule evaluation, deduplication, and notification
- [[downsampling-and-retention]] -- tiered data retention strategies